Each post's text is turned into a vector of 384 numbers with the sentence encoder `all-MiniLM-L6-v2`, which places posts with similar meaning close together. The vectors are grouped into `K=40` clusters with **e-means**. The clusters are then read and merged by hand into about a dozen named categories, and every post gets a category in `data/clean/post_category.csv`.

In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from faiss.contrib.e_means import NumpyEmeans
from sklearn.feature_extraction.text import CountVectorizer

K = 40  # number of clusters
EMBEDDINGS_PATH = Path("../data/embeddings.npy")  # saved embeddings path

Links, emoji placeholders and @mentions are removed, because they say nothing about the topic. Identical texts are embedded once, because a ReTruth repeats the text of the post it reposts -> every post then gets the cluster of its text. Posts with no words left (only a link, an emoji or media) get a specific category later.

In [2]:
posts = pd.read_csv("../data/clean/truths.csv", usecols=["id", "text"], keep_default_na=False, na_values=[""])
posts["clean"] = (posts.text.fillna("").str.replace(r"https?://\S+|<emoji: [^>]*>|@\w+", " ", regex=True).str.split().str.join(" "))
docs = posts.clean[posts.clean != ""].drop_duplicates().tolist()
print(f"{len(posts):,} posts, {len(docs):,} distinct texts to cluster")

929,592 posts, 603,135 distinct texts to cluster


`normalize_embeddings=True` gives every vector length 1, so comparing two vectors measures only the angle between them (cosine similarity), which is the usual way to compare sentence embeddings. The vectors are saved in half precision (`float16`), which halves the file size without a visible effect on the clusters.

In [3]:
if not EMBEDDINGS_PATH.exists():
    model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    start = time.time()
    vectors = model.encode(docs, batch_size=128, normalize_embeddings=True, show_progress_bar=True)
    np.save(EMBEDDINGS_PATH, vectors.astype("float16"))
    print(f"embedded in {time.time() - start:.0f} s")
X = np.load(EMBEDDINGS_PATH).astype("float32")
X.shape

(603135, 384)

e-means reads the data in small random batches. After each batch, every centroid moves a little towards the mean of the batch points assigned to it. The centroid is an exponential moving average, and the step size shrinks during training on a cosine curve. 
Every five batches, clusters that have almost no points are restarted by splitting a large cluster in two. 

`spherical=True` keeps the centroids at length 1, so posts are grouped by cosine similarity, matching the normalised embeddings.

In [4]:
em = NumpyEmeans(X.shape[1], K, spherical=True)
_, em_labels = em.train(X)

Each cluster is described by its most distinctive words and by the three texts closest to its centre, which are its most typical texts.

The distinctive words come from c-TF-IDF, the cluster version of TF-IDF.
The word counts of all the texts in a cluster are added up. A word's score is its share of the cluster's words, multiplied by log(1 + A / f), where f is how often the word occurs in the whole data and A is the average number of words in a cluster. A word therefore scores high when it is frequent in the cluster and rare overall.

In [5]:
cv = CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{3,}\b", min_df=20)
counts = cv.fit_transform(docs)  # one row per text, one column per word
words = cv.get_feature_names_out()
per_cluster = np.array([np.asarray(counts[em_labels == j].sum(axis=0)).ravel() for j in range(K)])

# c-TF-IDF
tf = per_cluster / per_cluster.sum(axis=1, keepdims=True)
idf = np.log(1 + per_cluster.sum(axis=1).mean() / per_cluster.sum(axis=0))
weights = tf * idf


summary = pd.DataFrame({
    "size": np.bincount(em_labels, minlength=K),
    "top words": [", ".join(words[row.argsort()[::-1][:8]]) for row in weights]
})
summary.sort_values("size", ascending=False)

,size,top words
17,36690,"maga, trump, truth, americafirst, saveamerica,..."
35,25377,"status, mention, class, link, url, hashtag, tr..."
9,23432,"trump, president, donald, america, maga, count..."
29,22478,"biden, joe, president, trump, administration, ..."
5,20231,"fbi, trump, secret, raid, cia, doj, news, evid..."
26,19964,"media, news, people, left, speech, like, censo..."
37,19799,"guy, man, like, joe, just, know, did, think"
8,19594,"america, country, government, world, people, w..."
2,19250,"inflation, money, gas, prices, market, tax, pr..."
1,18883,"yes, let, right, love, lol, time, good, agree"


In [6]:
GROUPS = {
    "Trump and MAGA": [17, 9, 35],
    "Biden and Harris": [29, 10],
    "Democrats and Republicans": [32, 12],
    "Elections and endorsements": [30, 34],
    "FBI, courts and trials": [5, 18],
    "Economy": [2],
    "Wars and foreign policy": [20, 8],
    "Vaccines and COVID": [16],
    "Immigration, crime and guns": [21, 4, 15],
    "Media, schools and gender": [26, 36],
    "Religion": [25],
    "Entertainment, sports and lifestyle": [28, 11, 38, 31, 24],
    "Truth Social and follows": [3, 14, 23, 39],
    "Reactions and chat": [1, 27, 22, 7, 37, 6, 0, 13, 33, 19],
}
NAMES = {c: name for name, clusters in GROUPS.items() for c in clusters}

has_text = posts.clean != ""
posts["cluster"] = posts.clean.map(dict(zip(docs, em_labels))).astype("Int64")
posts["category"] = posts.cluster.map(lambda c: NAMES.get(c, f"cluster {c}"), na_action="ignore")
posts.loc[~has_text, "category"] = np.where(posts.text[~has_text].str.contains("http", na=False),"Link only", "No text")

posts[["id", "cluster", "category"]].rename(columns={"id": "truth_id"}).to_csv("../data/clean/post_category.csv", index=False)
posts.category.value_counts()

category
Reactions and chat                     219872
Trump and MAGA                         119355
Entertainment, sports and lifestyle     67797
Truth Social and follows                63977
Link only                               63645
FBI, courts and trials                  51270
No text                                 45101
Wars and foreign policy                 42837
Media, schools and gender               40843
Biden and Harris                        40833
Elections and endorsements              40271
Democrats and Republicans               37573
Immigration, crime and guns             36652
Economy                                 23362
Religion                                19500
Vaccines and COVID                      16704
Name: count, dtype: int64